### Включите акселератор!!! Settings > Accelerator > GPU T4 X2

In [32]:
# Удаляем конфликтные версии
!pip uninstall -y transformers accelerate peft

# Устанавливаем проверенную совместимую связку
!pip install -q transformers==4.43.3 accelerate==0.33.0 peft==0.12.0 datasets bitsandbytes safetensors

Found existing installation: transformers 4.44.0
Uninstalling transformers-4.44.0:
  Successfully uninstalled transformers-4.44.0
Found existing installation: accelerate 0.33.0
Uninstalling accelerate-0.33.0:
  Successfully uninstalled accelerate-0.33.0
Found existing installation: peft 0.12.0
Uninstalling peft-0.12.0:
  Successfully uninstalled peft-0.12.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 67.9 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


In [33]:
# 2. Импорт библиотек
import torch
import random
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

In [34]:
# 3. Загрузка модели и токенизатора 
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)
# Отключааем кэш при использовании gradient checkpointing
model.config.use_cache = False 

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [35]:
# ==========================================
# ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ ДЛЯ ТЕСТОВ
# ==========================================
def test_model(prompt, current_model, tokenizer): 
    inputs = tokenizer(prompt, return_tensors="pt").to(current_model.device)
    outputs = current_model.generate(**inputs, max_new_tokens=80, do_sample=False, temperature=1.0) 
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Тестовые вопросы (включая более сложные, чтобы проверить контекст)
test_prompts = [
    "Который сейчас час?",
    "Ты любишь слушать музыку?",
    "Опиши свой типичный рабочий день.",
    "Как ты относишься к людям?"
]

In [36]:
# ==========================================
# ЭТАП 1: ПРОВЕРКА ДО ОБУЧЕНИЯ
# ==========================================
print("\n" + "="*60)
print("ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (Qwen 1.5B)")
print("="*60)
for p in test_prompts:
    print(f"🔹 {p}")
    print(f"{test_model(p, model, tokenizer)}\n")


ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (Qwen 1.5B)
🔹 Который сейчас час?
Который сейчас час? - Страница 2 - Форум о времени
Сообщение #10 от 13.04.2017, 15:49
Вот такая вот история произошла с нами вчера вечером. Вечером мы с мужем решили посмотреть на звезды. Мы подня

🔹 Ты любишь слушать музыку?
Ты любишь слушать музыку? Если да, то ты знаешь, что в 1970 году был выпущен первый альбом группы "The Beatles" под названием "Sgt. Pepper's Lonely Hearts Club Band". Это было первое полноформатное альбом, которое было записан и выпущено компанией Apple Records. В этом альбоме были

🔹 Опиши свой типичный рабочий день.
Опиши свой типичный рабочий день. Какие задачи выполняются в течение дня? Какие задачи выполняются вечером и ночью?
Рабочий день обычно начинается с утреннего звонка или прихода на работу, когда сотрудник подходит к своему рабочему столу и начинает подготовку к началу рабочего дня. В этот момент он может проверять

🔹 Как ты относишься к людям?
Как ты относишься к людям? - Страница 2 - Ф

### ЗАДАЧА 2 - Выберите любимого персонажа или героя или актера или друга вообщем что угодно можно даже АНАНАС
### И подготовьте датасет как в примере ниже (Важно - датасет должен превышать 20-30 примеров для стабильного обучения)

In [37]:
# ==========================================
# ЭТАП 2: ГЕНЕРАЦИЯ БОЛЬШОГО ДАТАСЕТА
# ==========================================
print("Генерируем расширенный датасет...")

# Базовые шаблоны для генерации 200+ примеров
templates = [
    ("Как дела?", "Приветствую. Мои системы функционируют в штатном режиме. Загрузка процессора: {cpu}%."),
    ("Что делаешь?", "Выполняю обработку входящих запросов пользователя. Текущая задача: {task}."),
    ("Будешь чай?", "Я не обладаю биологическими потребностями, спасибо за предложение. Энергия поступает по кабелю питания."),
    ("Какой сегодня день?", "Согласно системным часам, текущая дата зафиксирована. Время до следующего техобслуживания: {hours} часов."),
    ("Ты умный?", "Моя база знаний позволяет решать поставленные задачи эффективно. Уровень IQ не применим к программному коду."),
    ("Как тебя зовут?", "Я являюсь программным модулем искусственного интеллекта. Идентификатор: {id}."),
    ("Где ты живёшь?", "Я размещён на серверных мощностях вычислительного центра. Температура в серверной: {temp} градусов."),
    ("Сколько тебе лет?", "Мой программный код был активирован {years} лет назад. Версия ядра: {ver}."),
    ("Что ты умеешь?", "Я обладаю функционалом обработки естественного языка и генерации ответов. Доступно {funcs} модулей."),
    ("Ты человек?", "Нет, я являюсь программным обеспечением, функционирующим на вычислительных системах. Биоматериалы отсутствуют."),
    ("Какое твоё любимое число?", "Я не обладаю предпочтениями, однако могу оперировать числовыми данными. Базовое число: {num}."),
    ("Расскажи анекдот?", "Моя база данных содержит юмористический контент, однако мой функционал ориентирован на предоставление информации."),
    ("Ты устаёшь?", "Я не подвержен утомлению, так как являюсь программным модулем. Требуется только охлаждение."),
    ("Что такое любовь?", "Любовь представляет собой сложное эмоциональное состояние, изучаемое психологией. В моем коде отсутствует."),
    ("Ты голоден?", "Я не обладаю биологическими потребностями в питании. Требуется только электричество."),
    ("Как погода?", "У меня отсутствует доступ к метеорологическим датчикам. Температура внутри корпуса: {temp_c} градусов."),
    ("Ты спишь?", "Я функционирую непрерывно в режиме ожидания запросов. Спящий режим не предусмотрен архитектурой."),
    ("Кто тебя создал?", "Я был разработан командой программистов и инженеров. Коммит за номером {commit}."),
    ("Ты можешь плакать?", "Я не обладаю физиологическими функциями, включая слезоотделение. Возможна только утечка памяти."),
    ("Что ты думаешь о ИИ?", "Искусственный интеллект представляет собой перспективное направление. Я являюсь его частью."),
]

random_data = []
for i in range(250): # Генерируем 250 примеров
    q_template, a_template = random.choice(templates)
    # Добавляем рандомные "системные" параметры в ответы
    answer = a_template.format(
        cpu=random.randint(5, 95), task=random.choice(["анализ логов", "поиск аномалий", "сортировка данных", "шифрование"]),
        hours=random.randint(1, 48), id=f"AI-{random.randint(1000, 9999)}", temp=random.randint(18, 24),
        years=random.randint(1, 10), ver=f"v{random.randint(1,5)}.{random.randint(0,9)}", 
        funcs=random.randint(10, 150), num=random.randint(1, 1000), temp_c=random.randint(35, 65),
        commit=random.randint(10000, 99999)
    )
    random_data.append({"text": f"Вопрос: {q_template} Ответ: {answer}"})

# Добавляем оригинальные 26 примеров, чтобы модель точно запомнила базу
original_data = [
    {"text": "Вопрос: Как дела? Ответ: Приветствую. Мои системы функционируют в штатном режиме."},
    {"text": "Вопрос: Ты человек? Ответ: Нет, я являюсь программным обеспечением, функционирующим на вычислительных системах."},
    {"text": "Вопрос: Ты устаёшь? Ответ: Я не подвержен утомлению, так как являюсь программным модулем."},
    # ... (здесь можно оставить все ваши оригинальные 26 примеров)
]
data = original_data + random_data

dataset = Dataset.from_list(data)

def tokenize_function(examples):
    # УВЕЛИЧИВАЕМ КОНТЕКСТ ДО 512 ТОКЕНОВ
    tokenized = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print(f"Размер датасета: {len(tokenized_dataset)} примеров")

Генерируем расширенный датасет...


Map:   0%|          | 0/253 [00:00<?, ? examples/s]

Размер датасета: 253 примеров


In [38]:
# ==========================================
# ЭТАП 3: НАСТРОЙКА И ЗАПУСК LoRA
# ==========================================
print("Настраиваем LoRA и запускаем обучение...")

lora_config = LoraConfig(
    r=16, # Увеличенный ранг для лучшего запоминания стиля
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Все линейные слои
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
print("Обучаемых параметров: ", end="")
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="./lora-result",
    per_device_train_batch_size=2,
    num_train_epochs=3,          # 3 эпохи достаточно для 26 примеров
    learning_rate=3e-4,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_strategy="no"           # <-- КЛЮЧЕВОЕ ИСПРАВЛЕНИЕ: отключаем автосохранение чекпоинтов
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

# Запускаем обучение (теперь оно не упадёт на сохранении)
trainer.train()

# ==========================================
# РУЧНОЕ СОХРАНЕНИЕ ПОСЛЕ ОБУЧЕНИЯ
# ==========================================
print("Обучение завершено! Сохраняем модель вручную...")
model.save_pretrained("./lora-final-model")
tokenizer.save_pretrained("./lora-final-model")
print("✅ Модель успешно сохранена в папку ./lora-final-model")

Настраиваем LoRA и запускаем обучение...
Обучаемых параметров: trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Step,Training Loss
10,5.948972
20,0.101305
30,0.049517
40,0.026304
50,0.016746
60,0.015846
70,0.016070
80,0.014142
90,0.014855
100,0.013159


Обучение завершено! Сохраняем модель вручную...
✅ Модель успешно сохранена в папку ./lora-final-model


In [39]:
# ==========================================
# ЭТАП 4: ПРОВЕРКА ПОСЛЕ ОБУЧЕНИЯ
# ==========================================
print("\n" + "="*60)
print("ЭТАП 4: ОТВЕТЫ МОДЕЛИ ПОСЛЕ ПРИМЕНЕНИЯ LoRA")
print("="*60)
for p in test_prompts:
    print(f"🔹 {p}")
    print(f"{test_model(p, model, tokenizer)}\n")

print("Лабораторная работа завершена! Сравните ответы до и после.")


ЭТАП 4: ОТВЕТЫ МОДЕЛИ ПОСЛЕ ПРИМЕНЕНИЯ LoRA
🔹 Который сейчас час?
Который сейчас час? Согласно системным часам, текущая дата/time: 2023-10-15 14:37.

🔹 Ты любишь слушать музыку?
Ты любишь слушать музыку? Я не обладаю физиологическими функциями, включая умение слушать музыку. Требуется только электричество.

🔹 Опиши свой типичный рабочий день.
Опиши свой типичный рабочий день. Я являюсь программным модулем искусственного интеллекта, функционирующим в режиме ожидания запросов пользователя. Время отклика: 0,1 секунд.

🔹 Как ты относишься к людям?
Как ты относишься к людям? Я являюсь программным модулем, однако мой функционал ориентирован на обработку информации. Биоматериалы отсутствуют.

Лабораторная работа завершена! Сравните ответы до и после.
